### SoRL Playground — v3 + Anchor Loss

This notebook uses `SoRLTrainerv3` with anchor loss to observe:
1. **Inner monologue** — how abstract tokens interleave with natural language
2. **Loss curves** — all loss components over training

In [ ]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from sorl.trainer_ablate import SoRLTrainerv3, SoRLConfig
from data.pt_dataset import get_dataset, evaluate_accuracy, collate_fn, _filter_traj_tokens

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
# Initialize SoRL model
from sorl.sorl_wrapper import SorlModelWrapper
model_name = "Qwen/Qwen3-0.6B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name) 

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [ ]:
# ============================================================
# SoRL Training with SoRLTrainerv3 + Anchor Loss
# ============================================================

# Datasets
train_ds = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=256)
val_ds = get_dataset("gsm8k", split="test", tokenizer=tokenizer, max_length=256)

# Config — v3 contrastive + anchor loss
config = SoRLConfig(
    num_rollouts=4,
    K=4,
    max_iterations=2,
    memory_span_abs=1792,
    memory_span_traj=1792,
    temperature=1.0,
    # v3 loss weights
    alpha_traj=1.0,
    alpha_contrastive=1.0,
    gamma_contrastive=0.5,
    corrupt_method="shuffle",
    corrupt_ratio=0.3,
    alpha_abs=0.5,
    alpha_soft_zipf=2.0,
    alpha_ortho=0.0,
    alpha_anchor=1.0,          # <-- anchor loss enabled
    # optimizer
    lr=1e-5,
    emb_lr_mult=10.0,
    weight_decay=0.01,
    warmup_steps=50,
    cooldown_frac=0.4,
    max_grad_norm=1.0,
    # training
    batch_size=2,
    gradient_accumulation_steps=4,
    num_epochs=3,
    log_every=10,
    eval_every=99999,
    save_every=99999,
    output_dir="./ckpt/sorl_v3_anchor_playground",
)

trainer = SoRLTrainerv3(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    val_dataset=val_ds,
    compute_accuracy=evaluate_accuracy,
    collate_fn=collate_fn,
    config=config,
    device=str(device),
)
print("Trainer ready: SoRLTrainerv3 with anchor_loss")

In [4]:
trainer.train()

Total steps: 3737 | Steps/epoch: 3737 | Effective batch: 2
[emb_warmup] Froze all except abstract emb/proj rows. Trainable: 129 × 896 × 2 = 0.23M


KeyError: 'ortho_loss'

In [ ]:
# ============================================================
# Plot Loss Curves
# ============================================================
h = trainer.history

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("SoRLTrainerv3 + Anchor Loss — Training Curves", fontsize=14)

plots = [
    ("loss", "Total Loss"),
    ("base_loss", "Base Traj Loss (SFT equiv, no grad)"),
    ("traj_loss", "Traj Loss p(s|a)"),
    ("hinge_loss", "Hinge Contrastive Loss"),
    ("abs_loss", "Abstract Loss"),
    ("anchor_loss", "Anchor Loss"),
]

for ax, (key, title) in zip(axes.flat, plots):
    if key in h and len(h[key]) > 0:
        ax.plot(h["step"], h[key], linewidth=0.8)
        ax.set_title(title)
        ax.set_xlabel("step")
        ax.grid(True, alpha=0.3)
    else:
        ax.set_title(f"{title} (no data)")

plt.tight_layout()
plt.show()

# Also plot zipf + ortho if nonzero
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
for ax, (key, title) in zip(axes2, [("zipf_loss", "Zipf Loss"), ("ortho_loss", "Ortho Loss")]):
    if key in h and len(h[key]) > 0:
        ax.plot(h["step"], h[key], linewidth=0.8)
        ax.set_title(title)
        ax.set_xlabel("step")
        ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Inner Monologue Visualization
# ============================================================
# Generate from a few val samples and show how abstract tokens
# interleave with natural language tokens.

from sorl.sorl_trainer import sorl_search

model.eval()
base_vocab = int(model.vocab_sizes[0].item())
n_samples = 3
max_new_tokens = 128

for i in range(min(n_samples, len(val_ds))):
    item = val_ds[i]
    input_ids = item["input_ids"].unsqueeze(0).to(device)
    attention_mask = item["attention_mask"].unsqueeze(0).to(device)
    prompt_len = item["prompt_len"]

    # Decode question
    question = tokenizer.decode(input_ids[0, :prompt_len], skip_special_tokens=True)

    # Generate WITH abstract tokens (K=4)
    with torch.no_grad():
        generated = model.generate(
            input_ids=input_ids[:, :prompt_len],
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            K=config.K,
        )

    # Build annotated string: NL tokens as text, abstract tokens as [A_id]
    gen_tokens = generated[0, prompt_len:]
    parts = []
    for tok_id in gen_tokens:
        tid = tok_id.item()
        if tid == tokenizer.eos_token_id:
            break
        if tid >= base_vocab:
            parts.append(f"[A{tid - base_vocab}]")
        else:
            parts.append(tokenizer.decode([tid]))

    annotated = "".join(parts)

    # Also decode NL-only for comparison
    traj_tokens = _filter_traj_tokens(generated, base_vocab)
    nl_text = tokenizer.decode(traj_tokens[0][prompt_len:], skip_special_tokens=True)

    print(f"{'='*80}")
    print(f"Sample {i+1}")
    print(f"{'='*80}")
    print(f"Question: {question[:200]}...")
    print(f"\n--- Inner Monologue (with abstract tokens) ---")
    print(annotated[:500])
    print(f"\n--- NL-only output ---")
    print(nl_text[:500])
    print()

model.train()
print("Done.")

In [ ]:
# ============================================================
# Eval Accuracy (optional)
# ============================================================
model.eval()
result = evaluate_accuracy(
    model, tokenizer, val_ds,
    device, 5,
)
print(result)